In [1]:
# Импортируем необходимые библиотеки
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import shap
import seaborn as sns
import joblib, os

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

## Шаг: Загрузка, подготовка и исследование данных <a name="view"></a>

Считаем CSV-файлы с данными и сохраним их в датафреймы. 

In [2]:
try:
    purchases = pd.read_csv('data/apparel-purchases.csv')
    messages = pd.read_csv('data/apparel-messages.csv')
    campaigns = pd.read_csv('data/full_campaign_daily_event.csv')
    campaigns_ch = pd.read_csv('data/full_campaign_daily_event_channel.csv')
    target = pd.read_csv('data/apparel-target_binary.csv')
except:
    purchases = pd.read_csv('C:/marketing_data/apparel-purchases.csv')
    messages = pd.read_csv('C:/marketing_data/apparel-messages.csv')
    campaigns = pd.read_csv('C:/marketing_data/full_campaign_daily_event.csv')
    campaigns_ch = pd.read_csv('C:/marketing_data/full_campaign_daily_event_channel.csv')
    target = pd.read_csv('C:/marketing_data/apparel-target_binary.csv')

# Первичный анализ данных

In [3]:
def see_func (df):
    display(df.head(5))
    print(f'Размерность датафрейма: {df.shape[0]}')
    display(df.info()) 
    print(f'Количество дубликатов: {df.duplicated().sum()}')
    print(f'Количество пропусков: {df.isna().sum()}')

In [4]:
see_func(purchases)

,client_id,quantity,price,category_ids,date,message_id
0,1515915625468169594,1,1999.0,"['4', '28', '57', '431']",2022-05-16,1515915625468169594-4301-627b661e9736d
1,1515915625468169594,1,2499.0,"['4', '28', '57', '431']",2022-05-16,1515915625468169594-4301-627b661e9736d
2,1515915625471138230,1,6499.0,"['4', '28', '57', '431']",2022-05-16,1515915625471138230-4437-6282242f27843
3,1515915625471138230,1,4999.0,"['4', '28', '244', '432']",2022-05-16,1515915625471138230-4437-6282242f27843
4,1515915625471138230,1,4999.0,"['4', '28', '49', '413']",2022-05-16,1515915625471138230-4437-6282242f27843


Размерность датафрейма: 202208
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202208 entries, 0 to 202207
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   client_id     202208 non-null  int64  
 1   quantity      202208 non-null  int64  
 2   price         202208 non-null  float64
 3   category_ids  202208 non-null  object 
 4   date          202208 non-null  object 
 5   message_id    202208 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 9.3+ MB


None

Количество дубликатов: 73020
Количество пропусков: client_id       0
quantity        0
price           0
category_ids    0
date            0
message_id      0
dtype: int64


In [5]:
see_func(messages)

,bulk_campaign_id,client_id,message_id,event,channel,date,created_at
0,4439,1515915625626736623,1515915625626736623-4439-6283415ac07ea,open,email,2022-05-19,2022-05-19 00:14:20
1,4439,1515915625490086521,1515915625490086521-4439-62834150016dd,open,email,2022-05-19,2022-05-19 00:39:34
2,4439,1515915625553578558,1515915625553578558-4439-6283415b36b4f,open,email,2022-05-19,2022-05-19 00:51:49
3,4439,1515915625553578558,1515915625553578558-4439-6283415b36b4f,click,email,2022-05-19,2022-05-19 00:52:20
4,4439,1515915625471518311,1515915625471518311-4439-628341570c133,open,email,2022-05-19,2022-05-19 00:56:52


Размерность датафрейма: 12739798
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12739798 entries, 0 to 12739797
Data columns (total 7 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   bulk_campaign_id  int64 
 1   client_id         int64 
 2   message_id        object
 3   event             object
 4   channel           object
 5   date              object
 6   created_at        object
dtypes: int64(2), object(5)
memory usage: 680.4+ MB


None

Количество дубликатов: 48610
Количество пропусков: bulk_campaign_id    0
client_id           0
message_id          0
event               0
channel             0
date                0
created_at          0
dtype: int64


In [6]:
see_func(campaigns)

,date,bulk_campaign_id,count_click,count_complain,count_hard_bounce,count_open,count_purchase,count_send,count_soft_bounce,count_subscribe,...,nunique_open,nunique_purchase,nunique_send,nunique_soft_bounce,nunique_subscribe,nunique_unsubscribe,count_hbq_spam,nunique_hbq_spam,count_close,nunique_close
0,2022-05-19,563,0,0,0,4,0,0,0,0,...,4,0,0,0,0,0,0,0,0,0
1,2022-05-19,577,0,0,0,1,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,2022-05-19,622,0,0,0,2,0,0,0,0,...,2,0,0,0,0,0,0,0,0,0
3,2022-05-19,634,0,0,0,1,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,2022-05-19,676,0,0,0,1,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


Размерность датафрейма: 131072
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131072 entries, 0 to 131071
Data columns (total 24 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   date                 131072 non-null  object
 1   bulk_campaign_id     131072 non-null  int64 
 2   count_click          131072 non-null  int64 
 3   count_complain       131072 non-null  int64 
 4   count_hard_bounce    131072 non-null  int64 
 5   count_open           131072 non-null  int64 
 6   count_purchase       131072 non-null  int64 
 7   count_send           131072 non-null  int64 
 8   count_soft_bounce    131072 non-null  int64 
 9   count_subscribe      131072 non-null  int64 
 10  count_unsubscribe    131072 non-null  int64 
 11  nunique_click        131072 non-null  int64 
 12  nunique_complain     131072 non-null  int64 
 13  nunique_hard_bounce  131072 non-null  int64 
 14  nunique_open         131072 non-null  int64 
 15  nun

None

Количество дубликатов: 0
Количество пропусков: date                   0
bulk_campaign_id       0
count_click            0
count_complain         0
count_hard_bounce      0
count_open             0
count_purchase         0
count_send             0
count_soft_bounce      0
count_subscribe        0
count_unsubscribe      0
nunique_click          0
nunique_complain       0
nunique_hard_bounce    0
nunique_open           0
nunique_purchase       0
nunique_send           0
nunique_soft_bounce    0
nunique_subscribe      0
nunique_unsubscribe    0
count_hbq_spam         0
nunique_hbq_spam       0
count_close            0
nunique_close          0
dtype: int64


In [7]:
see_func(campaigns_ch)

,date,bulk_campaign_id,count_click_email,count_click_mobile_push,count_open_email,count_open_mobile_push,count_purchase_email,count_purchase_mobile_push,count_soft_bounce_email,count_subscribe_email,...,count_send_email,nunique_hard_bounce_email,nunique_hbq_spam_email,nunique_send_email,count_soft_bounce_mobile_push,nunique_soft_bounce_mobile_push,count_complain_email,nunique_complain_email,count_close_mobile_push,nunique_close_mobile_push
0,2022-05-19,563,0,0,4,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2022-05-19,577,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2022-05-19,622,0,0,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2022-05-19,634,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2022-05-19,676,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Размерность датафрейма: 131072
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131072 entries, 0 to 131071
Data columns (total 36 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   date                             131072 non-null  object
 1   bulk_campaign_id                 131072 non-null  int64 
 2   count_click_email                131072 non-null  int64 
 3   count_click_mobile_push          131072 non-null  int64 
 4   count_open_email                 131072 non-null  int64 
 5   count_open_mobile_push           131072 non-null  int64 
 6   count_purchase_email             131072 non-null  int64 
 7   count_purchase_mobile_push       131072 non-null  int64 
 8   count_soft_bounce_email          131072 non-null  int64 
 9   count_subscribe_email            131072 non-null  int64 
 10  count_unsubscribe_email          131072 non-null  int64 
 11  nunique_click_email              131072 non-nul

None

Количество дубликатов: 0
Количество пропусков: date                               0
bulk_campaign_id                   0
count_click_email                  0
count_click_mobile_push            0
count_open_email                   0
count_open_mobile_push             0
count_purchase_email               0
count_purchase_mobile_push         0
count_soft_bounce_email            0
count_subscribe_email              0
count_unsubscribe_email            0
nunique_click_email                0
nunique_click_mobile_push          0
nunique_open_email                 0
nunique_open_mobile_push           0
nunique_purchase_email             0
nunique_purchase_mobile_push       0
nunique_soft_bounce_email          0
nunique_subscribe_email            0
nunique_unsubscribe_email          0
count_hard_bounce_mobile_push      0
count_send_mobile_push             0
nunique_hard_bounce_mobile_push    0
nunique_send_mobile_push           0
count_hard_bounce_email            0
count_hbq_spam_email        

In [8]:
see_func(target)

,client_id,target
0,1515915625468060902,0
1,1515915625468061003,1
2,1515915625468061099,0
3,1515915625468061100,0
4,1515915625468061170,0


Размерность датафрейма: 49849
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49849 entries, 0 to 49848
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   client_id  49849 non-null  int64
 1   target     49849 non-null  int64
dtypes: int64(2)
memory usage: 779.0 KB


None

Количество дубликатов: 0
Количество пропусков: client_id    0
target       0
dtype: int64


# Предобработка данных

## Обработка категорий

In [9]:
# Функция для безопасного парсинга
def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

purchases['categories_list'] = purchases['category_ids'].apply(safe_eval)

# Извлекаем топ-50 категорий
all_cats = purchases.explode('categories_list')['categories_list'].value_counts().head(50).index.tolist()
top_cats = set(all_cats)

# One-hot для топ-категорий
cat_features = []
for cat in top_cats:
    col_name = f'cat_{cat}'
    purchases[col_name] = purchases['categories_list'].apply(lambda x: 1 if cat in x else 0)
    cat_features.append(col_name)


## Агрегация по клиентам

In [10]:
# Признаки из покупок
client_purch = purchases.groupby('client_id').agg(
    total_quantity=('quantity', 'sum'),
    avg_price=('price', 'mean'),
    n_purchases=('client_id', 'count'),
    n_unique_categories=('categories_list', lambda x: len(set('|'.join(map(str, x)).split('|'))))
).reset_index()

# Признаки из сообщений
client_msgs = messages.groupby('client_id').agg(
    total_messages=('message_id', 'count'),
    opened_rate=('event', lambda x: (x == 'opened').mean()),
    purchased_from_msg=('event', lambda x: (x == 'purchased').sum()),
    n_unique_campaigns=('bulk_campaign_id', 'nunique')
).reset_index()

## Объединение

In [11]:
# Объединяем всё по client_id
X = target[['client_id']].merge(client_purch, on='client_id', how='left')
X = X.merge(client_msgs, on='client_id', how='left')

# Заполняем NaN
X = X.fillna(0)

# Отделяем целевую переменную
y = target.set_index('client_id').loc[X['client_id']]['target'].values

# Удаляем client_id из признаков
feature_cols = [col for col in X.columns if col != 'client_id']
X_features = X[feature_cols]

# ШАГ 3: Обучение модели

Метрика: ROC AUC → используем eval_metric='AUC' в CatBoost

In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = []

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    early_stopping_rounds=100,
    verbose=False,
    random_seed=42
)

for train_idx, val_idx in cv.split(X_features, y):
    X_tr, X_val = X_features.iloc[train_idx], X_features.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
    pred_proba = model.predict_proba(X_val)[:, 1]
    auc_scores.append(roc_auc_score(y_val, pred_proba))

print(f"ROC AUC на CV: {np.mean(auc_scores):.4f}")

ROC AUC на CV: 0.6849


 # ШАГ 4: Финальное обучение и submission.csv

In [13]:
# Обучаем на всех данных
final_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    early_stopping_rounds=100,
    verbose=False
)
final_model.fit(X_features, y)

# Сохраняем
os.makedirs('models', exist_ok=True)
joblib.dump(final_model, 'models/marketing_model.pkl')

# по ТЗ: submission.csv = client_id + target_proba для всех клиентов из target без target
test_ids = target[['client_id']]
test_preds = final_model.predict_proba(X_features)[:, 1]

submission = pd.DataFrame({
    'client_id': test_ids['client_id'],
    'target_proba': test_preds
})
submission.to_csv('submission.csv', index=False)

## Проверка результата

In [14]:
#try:
    #data = pd.read_csv('/data/submission.csv',)
#except:
    #data = pd.read_csv('C:/marketing_data/submission.csv')